In [ ]:
!curl -L -o data/input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: Failed to open the file data/input.txt: No such file or directory
  0 1089k    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (56) Failure writing output to destination, passed 16375 returned 4294967295


In [1]:
with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
len(text), text[:999]

(1115394,
 "First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
#tockenize 
# map chars to integers

stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s] #encoder: take a str, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) #decode: take a list of int, and output a str

print(encode('hii there'))
print(decode(encode('hii there')))

#tradeoff between codebook size and sequence length
#there has 65 codebook size, but a larger sequence length
#gpt's BPE has a large size, but a small sequence lengtg

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [5]:
import tiktoken
enc = tiktoken.get_encoding('gpt2')
print(enc.n_vocab)
print(enc.encode('hii there'))

50257
[71, 4178, 612]


In [6]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
data.shape, data.dtype, data.ndim, data[:100]

(torch.Size([1115394]),
 torch.int64,
 1,
 tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
         53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
          1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
         57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
          6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
         58, 47, 64, 43, 52, 10,  0, 37, 53, 59]))

In [7]:
# split train, val
n = int(0.9 * len(data))
train = data[:n]
val = data[n:]
len(train), len(val)

(1003854, 111540)

In [8]:
# when train with transformer, we work with chunks of dataset to train, 
# bascally, sample random little chunks of the training set 
# chunks has length, maximum length (block_size)

#time dimension
block_size = 8
train[:block_size + 1] 

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train[:block_size]
y = train[1:block_size+1]

print('x', x)
print('y', y)

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context}, the target is {target}")
    
# so a block_size = 8 = 8 data sample, the model will learn the 9 sample. simutanitsly, in this way, when model do inference, 
# model can handel different lenght of context from 1 to block_size

x tensor([18, 47, 56, 57, 58,  1, 15, 47])
y tensor([47, 56, 57, 58,  1, 15, 47, 58])
when input is tensor([18]), the target is 47
when input is tensor([18, 47]), the target is 56
when input is tensor([18, 47, 56]), the target is 57
when input is tensor([18, 47, 56, 57]), the target is 58
when input is tensor([18, 47, 56, 57, 58]), the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]), the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), the target is 58


In [10]:
#batch dimension 
#when we do sampleing, we will have multiple batches of multiple chunks of text, they are all stack in single tensor
#for efficiency, let gpu busy, process multiple chunks all at the same time

torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split:str):
    data = train if split == 'train' else val
    ix = torch.randint(len(data) - block_size, (batch_size, )) # e.g. tensor([459831, 425370, 157902, 365622])
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(f"inputs: {xb.shape}")
print(xb)

print(f"targets: {yb.shape}")
print(yb)


inputs: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets: torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [11]:
torch.randint(len(data) - block_size, (batch_size,))

tensor([1081275,  241627,  748567,  905830])

In [12]:
for b in range(batch_size):      #batch dimension
    for t in range(block_size):  #time dimension
        context = xb[b, :t + 1]
        target = yb[b, t]
        print(f"when input is {context}, the target is {target}")

when input is tensor([24]), the target is 43
when input is tensor([24, 43]), the target is 58
when input is tensor([24, 43, 58]), the target is 5
when input is tensor([24, 43, 58,  5]), the target is 57
when input is tensor([24, 43, 58,  5, 57]), the target is 1
when input is tensor([24, 43, 58,  5, 57,  1]), the target is 46
when input is tensor([24, 43, 58,  5, 57,  1, 46]), the target is 43
when input is tensor([24, 43, 58,  5, 57,  1, 46, 43]), the target is 39
when input is tensor([44]), the target is 53
when input is tensor([44, 53]), the target is 56
when input is tensor([44, 53, 56]), the target is 1
when input is tensor([44, 53, 56,  1]), the target is 58
when input is tensor([44, 53, 56,  1, 58]), the target is 46
when input is tensor([44, 53, 56,  1, 58, 46]), the target is 39
when input is tensor([44, 53, 56,  1, 58, 46, 39]), the target is 58
when input is tensor([44, 53, 56,  1, 58, 46, 39, 58]), the target is 1
when input is tensor([52]), the target is 58
when input is t

In [13]:
#bigram language model

# P(x1​,…,xT​)=t=1∏T​P(xt​∣x<t​),x<1​=∅  P(A,B)=P(A)P(B∣A)..
# Markov Assumption (Approximation) P(xt​∣x1​,…,xt−1​) ≈ P(xt​∣xt−1​) 
# The chain rule is lossless; this is a modeling assumption that 
# discards information in exchange for tractability.

# Bigram Is Tractable 
# Size ∣V∣×∣V∣ — for nanoGPT's char-level vocab, 65×65=4,225 parameters. 

# Bigram: ≈(truncate history to stay tractable)
# Transformer: =(keep full history; attention makes it computable)

import torch
from torch import nn 
from torch.nn import functional as F
torch.manual_seed(1337)


In [23]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets):
        # idx and targets are both (B,T) tensor
        logits = self.token_embedding_table(idx) # (B,T,C)
        
        B, T, C = logits.shape
        #cross_entropy require:pred is 2d (N, C),labels is 1d (N,),N is the # sample 
        logits = logits.view(B*T, C)
        targets = targets.view(B*T) 
        loss = F.cross_entropy(logits, targets)
        
        return logits, loss

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


torch.Size([32, 65])
tensor(4.3603, grad_fn=<NllLossBackward0>)


In [24]:
import numpy as np 
-np.log(1/65)

4.174387269895638

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        #logits = onehot(idx) @ W
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensor
        logits = self.token_embedding_table(idx) # (B,T,C)
        
        if targets is None:
            loss = None
        else:    
            B, T, C = logits.shape
            #cross_entropy require:pred is 2d (N, C),labels is 1d (N,),N is the # sample 
            logits = logits.view(B*T, C)
            targets = targets.view(B*T) 
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # get the prediction  (B, T, C)
            logits = logits[:, -1, :] # become (B, C) Focus on the last time step
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) #(B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T + 1)
        return idx
            
            
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


torch.Size([32, 65])
tensor(4.6763, grad_fn=<NllLossBackward0>)


In [26]:
""" torch.multinomial?
Returns a tensor where each row contains :attr:`num_samples` indices sampled
from the multinomial (a stricter definition would be multivariate,
refer to torch.distributions.multinomial.Multinomial for more details)
probability distribution located in the corresponding row """

' torch.multinomial?\nReturns a tensor where each row contains :attr:`num_samples` indices sampled\nfrom the multinomial (a stricter definition would be multivariate,\nrefer to torch.distributions.multinomial.Multinomial for more details)\nprobability distribution located in the corresponding row '

In [ ]:
""" 
epoch 1 e.g.
[0, 2, 1, 1, 3, 0, 2, 1]   -> "acbbdacb"
[3, 3, 1, 0, 0, 2, 2, 1]   -> "ddbaaccb"
[1, 0, 3, 2, 1, 1, 0, 3]   -> "badcbbad" 

logits, loss = self(idx) → logits shape (B, T, C) = (3, 8, 4)

logits = logits[:, -1, :] → (B, C) = (3, 4)
  logits: [1.2, 0.3, 2.5, 0.1]
  logits: [0.5, 2.0, 0.4, 1.1]
  logits: [2.2, 0.1, 0.3, 0.9]
  
probs = F.softmax(logits, dim=-1) → (3, 4)
  0 probs: [0.20, 0.08, 0.68, 0.04]  
  1 probs: [0.11, 0.50, 0.10, 0.29]  
  2 probs: [0.66, 0.08, 0.10, 0.16]   
  
idx_next = torch.multinomial(probs, num_samples=1) → (B, 1) = (3, 1)
  0 -> 2 ('c')
  1 -> 1 ('b')
  2 -> 0 ('a')
idx_next = [[2],
            [1],
            [0]]     shape (3, 1)

idx = torch.cat((idx, idx_next), dim=1) → (B, T+1) = (3, 9)            
     -> "acbbdacb" + "c"
     -> "ddbaaccb" + "b"
     -> "badcbbad" + "a"
"""

In [110]:
idx = torch.zeros((1, 1), dtype = torch.long) #(B=1, T=1)
idxx = m.generate(idx, max_new_tokens=100) #(B=1, T=101)
print(idxx)
print(decode(idxx[0].tolist()))

tensor([[ 0, 39, 20, 47,  8, 43, 34, 41, 27, 53, 55, 33, 50, 45,  1, 54, 57, 23,
         50, 36, 13, 28, 36, 29, 60, 45, 40, 22, 53, 42, 34,  7, 62, 51, 13, 52,
         22, 19, 31, 60,  0, 29, 33, 54, 18, 54,  2,  3,  9,  2,  6,  2, 53, 51,
         30, 22,  0, 15, 15,  9, 26, 46, 51, 13, 25, 23, 32, 64, 50, 35, 14, 43,
          3,  2,  3, 60, 50, 45, 38, 48, 49, 47, 58, 50, 16, 15,  1, 53, 64, 10,
         18,  0, 26, 44, 24, 57, 51, 13, 30, 34, 43]])

aHi.eVcOoqUlg psKlXAPXQvgbJodV-xmAnJGSv
QUpFp!$3!,!omRJ
CC3NhmAMKTzlWBe$!$vlgZjkitlDC oz:F
NfLsmARVe


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) #take grad, and update parameters based on grad

In [120]:
batck_size = 32

for steps in range(10000):
    xb, yb = get_batch('train')
    
    logits, loss = m(xb, yb)    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.1915972232818604


In [125]:
idx = torch.zeros((1, 1), dtype = torch.long) #(B=1, T=1)
idxx = m.generate(idx, max_new_tokens=500) #(B=1, T=101)
print(decode(idxx[0].tolist()))




Hate, izeanil isimy te f thatile moncrurdaivek.
dilveng?
Bu win m.
PORKANI bstend thironcendJve showe andingel'
Amalathot iowhily me cer te s CLUEsthul miminot s harilene mpa an s,
keeser f he be OXQXEES: our dloumo hades momy tr s y gor'llllenme te
ABror y t, the cay bur n urs:
RS: berethecatou n
Entoue,'d thouon y towhe atsintomend. RAURKESoct.
Thond y st INGl, nt, thay
Ifof athear:
Ane chouthe pend. an arldend e; sis' wad, amy o lds hm t, wis mecrvee nen?
I det GHecosth CKnointau'd r he y-



In [162]:
#in biggram the pred is only based on the previous tocken 
# the train goal is also xt|xt-1
# so it's the same for the input like T=8, T=1, when they have the same previous tocken 

a = torch.tensor([[43, 43, 43, 43, 43, 43, 43, 43]], dtype = torch.long) #(1,8)
b = torch.tensor([[43]], dtype=torch.long)                        #(1,1)


torch.manual_seed(121); print(decode(m.generate(a, max_new_tokens=1)[0].tolist()))

torch.manual_seed(121); print(decode(m.generate(b, max_new_tokens=1)[0].tolist()))

eeeeeeeer
er


In [166]:
print(list(m.parameters()))


[Parameter containing:
tensor([[  2.0657, -11.6655, -10.6177,  ...,  -9.0152,  -1.9486,  -9.9599],
        [ -9.8516,  -4.1220, -10.9635,  ..., -10.8405,   0.5674,  -4.9782],
        [  2.3160,   1.9179,  -5.4680,  ...,  -4.5255,  -5.0288,  -4.7703],
        ...,
        [ -2.8329,   0.5075,  -2.8614,  ...,  -4.3650,  -0.9772,  -2.8810],
        [ -0.5363,   2.5336,  -1.3463,  ..., -10.2804,  -8.4891,  -8.9959],
        [ -2.2668,  -1.0972,  -1.4863,  ...,  -2.9403,  -0.2514,  -0.9570]],
       requires_grad=True)]
